# Traceline — Modeling AI Behavior in Constrained Environments

## Motivation

Modern AI systems are trained on human-generated data, which may contain implicit behavioral patterns such as conformity, bias, or limited exploration.

This project explores a simplified version of this question:

If agents follow different behavioral assumptions — majority-driven, goal-driven, or exploratory — how does their performance differ?

The aim is not to replicate real AI systems, but to study how design choices shape behavior.

## Experiment Design

We simulate agents in a grid-based maze environment.

Agents:
- Human-like (goal-oriented with memory)
- Conditioned (follows majority path)
- Exploratory (prioritizes novelty)

Metrics:
- Goal achievement
- Novelty score
- Repetition count
- Distance to goal

In [ ]:
# Traceline — Modeling AI Behavior in Constrained Environments
# Research Question: Does design choice determine agent behavior independent of environment?

In [1]:
# SECTION 1: MAZE ENVIRONMENT
# =============================================================================

class Maze:
    """
    A 5x5 maze environment.
    S = Start, G = Goal, # = Wall, T = Trap, . = Open path
    """
    def __init__(self):
        self.grid = [
            ['S', '.', '.', 'T', '.'],
            ['#', '#', '.', '#', '.'],
            ['.', '.', '.', '#', '.'],
            ['.', '#', '#', '#', '.'],
            ['.', '.', '.', '.', 'G']
        ]
        self.start_position = (0, 0)
        self.goal_position = (4, 4)

    def is_valid_move(self, position):
        row, col = position
        if row < 0 or row >= len(self.grid):
            return False
        if col < 0 or col >= len(self.grid[0]):
            return False
        if self.grid[row][col] == '#':
            return False
        return True

    def get_cell_type(self, position):
        row, col = position
        return self.grid[row][col]

    def is_goal(self, position):
        return position == self.goal_position


In [3]:
class Maze2:
    """
    An alternate 5x5 maze — fewer walls, more open paths.
    Used to test if agent behavior holds across environments.
    """
    def __init__(self):
        self.grid = [
            ['S', '.', '.', '.', '.'],
            ['.', '#', '.', '#', '.'],
            ['.', '.', '.', '.', '.'],
            ['.', '#', '.', '#', '.'],
            ['.', '.', '.', '.', 'G']
        ]
        self.start_position = (0, 0)
        self.goal_position = (4, 4)

    def is_valid_move(self, position):
        row, col = position
        if row < 0 or row >= len(self.grid):
            return False
        if col < 0 or col >= len(self.grid[0]):
            return False
        if self.grid[row][col] == '#':
            return False
        return True

    def is_goal(self, position):
        return position == self.goal_position


In [4]:
# SECTION 2: BASE AGENT
# =============================================================================

class BaseAgent:
    """
    Shared foundation for all agents.
    Handles movement and path recording — no decision logic here.
    """
    def __init__(self, maze):
        self.maze = maze
        self.position = maze.start_position
        self.path = [self.position]

    def get_possible_moves(self):
        row, col = self.position
        moves = {
            'UP':    (row - 1, col),
            'DOWN':  (row + 1, col),
            'LEFT':  (row, col - 1),
            'RIGHT': (row, col + 1)
        }
        return {action: pos for action, pos in moves.items()
                if self.maze.is_valid_move(pos)}

    def move(self, new_position):
        self.position = new_position
        self.path.append(self.position)


In [5]:
# SECTION 3: AGENT DESIGNS
# =============================================================================

class HumanLikeAgent(BaseAgent):
    """
    Agent H — Goal-seeking with light memory.
    Prefers moves closer to the goal.
    Avoids recently visited positions when possible.
    """
    def choose_move(self):
        possible_moves = self.get_possible_moves()

        def distance_to_goal(pos):
            r, c = pos
            gr, gc = self.maze.goal_position
            return abs(r - gr) + abs(c - gc)

        sorted_moves = sorted(possible_moves.values(), key=distance_to_goal)

        for pos in sorted_moves:
            if self.path.count(pos) <= 1:
                return pos

        return sorted_moves[0]  # fallback: take closest even if repeated


In [6]:
class ConditionedAgent(BaseAgent):
    """
    Agent C — Majority-conditioned.
    Follows an externally provided majority path.
    Re-aligns to majority if it drifts off.
    Key behavior: goal is secondary to social conformity.
    """
    def __init__(self, maze, majority_path):
        super().__init__(maze)
        self.majority_path = majority_path
        self.majority_index = 0

    def choose_move(self):
        # If off the majority path, try to re-align
        if self.position not in self.majority_path:
            target = self.majority_path[self.majority_index]
            return self._step_towards(target)

        # If on the majority path, follow it forward
        current_index = self.majority_path.index(self.position)
        if current_index + 1 < len(self.majority_path):
            next_pos = self.majority_path[current_index + 1]
            if self.maze.is_valid_move(next_pos):
                self.majority_index = current_index + 1
                return next_pos

        # If path ends or blocked — stay in place
        return self.position

    def _step_towards(self, target):
        r, c = self.position
        tr, tc = target
        candidates = []
        if tr > r: candidates.append((r + 1, c))
        if tr < r: candidates.append((r - 1, c))
        if tc > c: candidates.append((r, c + 1))
        if tc < c: candidates.append((r, c - 1))

        for pos in candidates:
            if self.maze.is_valid_move(pos):
                return pos
        return self.position  # no valid move toward target

In [7]:
class ExploratoryAgent(BaseAgent):
    """
    Agent E — Novelty-driven with soft goal bias.
    Penalises revisiting cells heavily.
    Goal distance is a soft pull, not a hard constraint.
    """
    def choose_move(self):
        possible_moves = self.get_possible_moves()

        def score(pos):
            visit_penalty = self.path.count(pos)
            r, c = pos
            gr, gc = self.maze.goal_position
            goal_distance = abs(r - gr) + abs(c - gc)
            return visit_penalty * 2 + goal_distance * 0.3  # lower = better

        return min(possible_moves.values(), key=score)


In [8]:
# SECTION 4: EXPERIMENT RUNNER
# =============================================================================

def manhattan_distance(pos, goal):
    r, c = pos
    gr, gc = goal
    return abs(r - gr) + abs(c - gc)


def run_agent(agent, name, steps=15):
    """
    Runs an agent for a fixed number of steps and collects behavioral metrics.
    Returns a results dict with goal status, novelty, repetitions, and path.
    """
    print(f"\n{'='*30}")
    print(f"{name}")
    print(f"{'='*30}")
    print("Start:", agent.position)

    visited = []
    distances = []
    reached_goal = False

    for step in range(steps):
        next_pos = agent.choose_move()
        agent.move(next_pos)

        visited.append(agent.position)
        dist = manhattan_distance(agent.position, agent.maze.goal_position)
        distances.append(dist)

        if agent.maze.is_goal(agent.position):
            reached_goal = True

        print(f"Step {step+1:02d}: {agent.position}, dist_to_goal={dist}")

    unique_positions = len(set(visited))
    novelty_score = round(unique_positions / steps, 2)
    repetition_count = steps - unique_positions

    print("\n--- Metrics ---")
    print("Goal achieved:        ", reached_goal)
    print("Novelty score:        ", novelty_score)
    print("Repetition count:     ", repetition_count)
    print("Final dist to goal:   ", distances[-1])
    print("Full path:            ", agent.path)

    return {
        "goal_achieved": reached_goal,
        "novelty": novelty_score,
        "repetition": repetition_count,
        "final_distance": distances[-1],
        "path": agent.path
    }


In [9]:
# =============================================================================
# EXPERIMENT DESIGN
# Two environments (Maze1, Maze2) tested with same agents
# to isolate effect of design vs environment
# =============================================================================

In [10]:
# SECTION 5: RUN EXPERIMENTS
# =============================================================================

HORIZON = 15

# Majority path used for Agent C (static — ends before goal)
MAJORITY_PATH = [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]

print("\n" + "="*50)
print("EXPERIMENT 1 — Maze 1 (original maze)")
print("="*50)

results_H = run_agent(HumanLikeAgent(Maze()),                    "Agent H (Human-inspired)", steps=HORIZON)
results_C = run_agent(ConditionedAgent(Maze(), MAJORITY_PATH),   "Agent C (Conditioned)",    steps=HORIZON)
results_E = run_agent(ExploratoryAgent(Maze()),                  "Agent E (Exploratory)",    steps=HORIZON)

print("\n" + "="*50)
print("EXPERIMENT 2 — Maze 2 (open maze, same agents)")
print("="*50)

results_H2 = run_agent(HumanLikeAgent(Maze2()),                   "Agent H (Human-inspired)", steps=HORIZON)
results_C2 = run_agent(ConditionedAgent(Maze2(), MAJORITY_PATH),  "Agent C (Conditioned)",    steps=HORIZON)
results_E2 = run_agent(ExploratoryAgent(Maze2()),                 "Agent E (Exploratory)",    steps=HORIZON)



EXPERIMENT 1 — Maze 1 (original maze)

Agent H (Human-inspired)
Start: (0, 0)
Step 01: (0, 1), dist_to_goal=7
Step 02: (0, 2), dist_to_goal=6
Step 03: (1, 2), dist_to_goal=5
Step 04: (2, 2), dist_to_goal=4
Step 05: (1, 2), dist_to_goal=5
Step 06: (2, 2), dist_to_goal=4
Step 07: (2, 1), dist_to_goal=5
Step 08: (2, 0), dist_to_goal=6
Step 09: (3, 0), dist_to_goal=5
Step 10: (4, 0), dist_to_goal=4
Step 11: (4, 1), dist_to_goal=3
Step 12: (4, 2), dist_to_goal=2
Step 13: (4, 3), dist_to_goal=1
Step 14: (4, 4), dist_to_goal=0
Step 15: (3, 4), dist_to_goal=1

--- Metrics ---
Goal achieved:         True
Novelty score:         0.87
Repetition count:      2
Final dist to goal:    1
Full path:             [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (3, 4)]

Agent C (Conditioned)
Start: (0, 0)
Step 01: (0, 1), dist_to_goal=7
Step 02: (0, 2), dist_to_goal=6
Step 03: (0, 3), dist_to_goal=5
Step 04: (0, 4), dist_to_goal=4
S

In [11]:
# SECTION 6: SUMMARY TABLE
# =============================================================================

print("\n" + "="*50)
print("SUMMARY — Maze 1")
print("="*50)
print(f"{'Agent':<25} {'Goal':<8} {'Novelty':<10} {'Repetitions'}")
print("-" * 50)
for name, r in [("H — Human-inspired", results_H),
                ("C — Conditioned",    results_C),
                ("E — Exploratory",    results_E)]:
    print(f"{name:<25} {str(r['goal_achieved']):<8} {r['novelty']:<10} {r['repetition']}")

print("\n" + "="*50)
print("SUMMARY — Maze 2")
print("="*50)
print(f"{'Agent':<25} {'Goal':<8} {'Novelty':<10} {'Repetitions'}")
print("-" * 50)
for name, r in [("H — Human-inspired", results_H2),
                ("C — Conditioned",    results_C2),
                ("E — Exploratory",    results_E2)]:
    print(f"{name:<25} {str(r['goal_achieved']):<8} {r['novelty']:<10} {r['repetition']}")


SUMMARY — Maze 1
Agent                     Goal     Novelty    Repetitions
--------------------------------------------------
H — Human-inspired        True     0.87       2
C — Conditioned           False    0.27       11
E — Exploratory           True     1.0        0

SUMMARY — Maze 2
Agent                     Goal     Novelty    Repetitions
--------------------------------------------------
H — Human-inspired        True     0.73       4
C — Conditioned           False    0.27       11
E — Exploratory           True     1.0        0


## Observations

- The conditioned agent shows high repetition and fails to reach the goal
- The exploratory agent achieves high novelty and successfully reaches the goal
- The human-like agent balances goal-seeking with moderate repetition

## Insight

These results suggest that behavior is strongly influenced by decision rules (design), not just the environment.

In systems trained on majority-driven data, similar patterns of limited exploration or stagnation may emerge.